# CosyVoice3 Fine-Tuning (Debi & Marlene)

## Overview
- Base Model: Fun-CosyVoice3-0.5B (Qwen2.5-0.5B LLM backbone)
- LLM SFT only (Flow/HiFiGAN pretrained weights 유지)
- Multi-speaker: debi + marlene (CAMPlus speaker embedding으로 분리)
- 4 style tags: [excited], [sad], [happy], [calm]

## Pipeline
1. Google Drive 마운트 + 데이터 업로드
2. CosyVoice repo clone + 환경 구축
3. Speaker Embedding 추출 (campplus.onnx)
4. Speech Token 추출 (speech_tokenizer_v3.onnx)
5. Parquet 변환
6. Dry Run (1 epoch 검증)
7. LLM SFT 학습
8. Checkpoint 평균화 + 평가
9. HuggingFace Hub 업로드

## Cell 1: Google Drive Mount + GPU Check

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# GPU 확인
!nvidia-smi

# 작업 디렉토리 설정
import os
WORK_DIR = '/content/CosyVoice'
DATA_DIR = '/content/cosyvoice3_data'
BACKUP_DIR = '/content/drive/MyDrive/cosyvoice3_backup'

os.makedirs(BACKUP_DIR, exist_ok=True)
print(f'Work dir: {WORK_DIR}')
print(f'Data dir: {DATA_DIR}')
print(f'Backup dir: {BACKUP_DIR}')

## Cell 2: 데이터 업로드

로컬에서 `prepare_cosyvoice3_data.py`로 생성한 `cosyvoice3_data/` 폴더를
Google Drive에 업로드한 후 여기서 복사합니다.

**또는** zip 파일을 Colab에 직접 업로드해도 됩니다.

In [ ]:
import shutil

# Option A: Google Drive에서 복사
DRIVE_DATA = '/content/drive/MyDrive/cosyvoice3_data'

if os.path.exists(DRIVE_DATA):
    if os.path.exists(DATA_DIR):
        shutil.rmtree(DATA_DIR)
    shutil.copytree(DRIVE_DATA, DATA_DIR)
    print(f'Google Drive에서 복사 완료: {DATA_DIR}')
else:
    print(f'{DRIVE_DATA} 없음 - zip 업로드를 사용하세요')
    # Option B: zip 업로드
    # from google.colab import files
    # uploaded = files.upload()  # cosyvoice3_data.zip 업로드
    # !unzip -q cosyvoice3_data.zip -d /content/

# 데이터 확인
for split in ['train', 'dev']:
    split_dir = os.path.join(DATA_DIR, split)
    if os.path.exists(split_dir):
        wav_count = len([f for f in os.listdir(os.path.join(split_dir, 'wavs')) if f.endswith('.wav')])
        print(f'{split}: {wav_count} wavs')
        # text 파일 샘플 확인
        with open(os.path.join(split_dir, 'text'), 'r') as f:
            lines = f.readlines()
            print(f'  text lines: {len(lines)}')
            print(f'  sample: {lines[0].strip()[:100]}...')

## Cell 3: CosyVoice Repo Clone + 환경 구축

Docker 대신 직접 clone + conda 환경을 구축합니다.
`neosun/cosyvoice:v3.4.0`은 추론 전용이라 학습에는 공식 repo가 필요합니다.

In [ ]:
%%bash
# CosyVoice repo clone
cd /content
if [ ! -d "CosyVoice" ]; then
    git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git
    cd CosyVoice
    git submodule update --init --recursive
else
    echo "CosyVoice already cloned"
fi

In [ ]:
%%bash
# 의존성 설치 (Colab에는 이미 PyTorch + CUDA 있음)
cd /content/CosyVoice

pip install -q conformer==0.3.2 \
    deepspeed==0.14.5 \
    diffusers==0.32.2 \
    gdown \
    grpcio==1.63.0 grpcio-tools==1.63.0 \
    hydra-core==1.3.2 \
    HyperPyYAML==1.2.2 \
    inflect==7.3.1 \
    librosa==0.10.2.post1 \
    lightning==2.4.0 \
    matplotlib==3.9.2 \
    modelscope==1.22.3 \
    networkx==3.4.2 \
    omegaconf==2.3.0 \
    onnxruntime-gpu==1.19.0 \
    openai-whisper==20240930 \
    protobuf==5.28.3 \
    pydantic==2.9.2 \
    rich==13.9.4 \
    soundfile==0.12.1 \
    tensorboard==2.18.0 \
    transformers==4.48.2 \
    wget==3.2

# WeTextProcessing (CosyVoice 텍스트 정규화)
pip install -q WeTextProcessing==1.0.4.1 || echo 'WeTextProcessing install failed - may need manual fix'

# matcha-tts (CosyVoice flow에서 필요)
cd /content/CosyVoice/third_party/Matcha-TTS
pip install -q -e . 2>/dev/null || echo 'Matcha-TTS install skipped'

echo 'Dependencies installed'

In [ ]:
%%bash
# Pretrained 모델 다운로드 (Fun-CosyVoice3-0.5B)
cd /content/CosyVoice

MODEL_DIR="pretrained_models/Fun-CosyVoice3-0.5B"
if [ ! -d "$MODEL_DIR" ]; then
    mkdir -p pretrained_models
    # HuggingFace에서 다운로드
    pip install -q huggingface_hub
    python -c "
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id='FunAudioLLM/Fun-CosyVoice3-0.5B-2512',
    local_dir='pretrained_models/Fun-CosyVoice3-0.5B',
    ignore_patterns=['*.md', '*.txt'],
)
print('Model downloaded')
"
else
    echo "Model already exists at $MODEL_DIR"
fi

# 파일 확인
ls -la $MODEL_DIR/

## Cell 4: 데이터 디렉토리 링크 설정

CosyVoice 학습 스크립트가 기대하는 위치에 데이터를 연결합니다.

In [ ]:
%%bash
cd /content/CosyVoice

# 학습 디렉토리 구성
TRAIN_DIR="examples/libritts/cosyvoice3"
mkdir -p $TRAIN_DIR/data

# cosyvoice3_data -> CosyVoice 학습 경로에 심볼릭 링크
ln -sfn /content/cosyvoice3_data/train $TRAIN_DIR/data/train
ln -sfn /content/cosyvoice3_data/dev $TRAIN_DIR/data/dev

# conf 디렉토리 확인 (cosyvoice3.yaml이 있어야 함)
if [ -f "$TRAIN_DIR/conf/cosyvoice3.yaml" ]; then
    echo "Config found: $TRAIN_DIR/conf/cosyvoice3.yaml"
else
    echo "Config not found - will create custom config"
fi

# 데이터 확인
echo '--- train ---'
wc -l $TRAIN_DIR/data/train/wav.scp $TRAIN_DIR/data/train/text
echo '--- dev ---'
wc -l $TRAIN_DIR/data/dev/wav.scp $TRAIN_DIR/data/dev/text

## Cell 5: Speaker Embedding 추출 (CAMPlus)

In [ ]:
%%bash
cd /content/CosyVoice

PRETRAINED="pretrained_models/Fun-CosyVoice3-0.5B"

for split in train dev; do
    echo "=== Extracting embeddings: $split ==="
    python tools/extract_embedding.py \
        --dir examples/libritts/cosyvoice3/data/$split \
        --onnx_path $PRETRAINED/campplus.onnx \
        --num_thread 4
    echo "Done: $split"
done

# 결과 확인
echo ''
echo '=== 결과 확인 ==='
for split in train dev; do
    DIR="examples/libritts/cosyvoice3/data/$split"
    ls -la $DIR/utt2embedding.pt $DIR/spk2embedding.pt 2>/dev/null || echo "$split: embedding files missing!"
done

## Cell 6: Speech Token 추출 (speech_tokenizer_v3)

In [ ]:
%%bash
cd /content/CosyVoice

PRETRAINED="pretrained_models/Fun-CosyVoice3-0.5B"

for split in train dev; do
    echo "=== Extracting speech tokens: $split ==="
    python tools/extract_speech_token.py \
        --dir examples/libritts/cosyvoice3/data/$split \
        --onnx_path $PRETRAINED/speech_tokenizer_v3.onnx
    echo "Done: $split"
done

# 결과 확인
echo ''
echo '=== 결과 확인 ==='
for split in train dev; do
    DIR="examples/libritts/cosyvoice3/data/$split"
    ls -la $DIR/utt2speech_token.pt 2>/dev/null || echo "$split: speech token file missing!"
done

## Cell 7: Parquet 변환

In [ ]:
%%bash
cd /content/CosyVoice

for split in train dev; do
    DIR="examples/libritts/cosyvoice3/data/$split"
    echo "=== Making parquet: $split ==="
    
    mkdir -p $DIR/parquet
    
    python tools/make_parquet_list.py \
        --num_utts_per_parquet 1000 \
        --num_processes 4 \
        --src_dir $DIR \
        --des_dir $DIR/parquet
    
    echo "Done: $split"
    echo "Parquet files:"
    ls -la $DIR/parquet/
done

# data.list 생성 (학습에 필요)
TRAIN_BASE="examples/libritts/cosyvoice3"
cat $TRAIN_BASE/data/train/parquet/data.list > $TRAIN_BASE/data/train.data.list
cat $TRAIN_BASE/data/dev/parquet/data.list > $TRAIN_BASE/data/dev.data.list

echo ''
echo 'train.data.list:'
cat $TRAIN_BASE/data/train.data.list
echo ''
echo 'dev.data.list:'
cat $TRAIN_BASE/data/dev.data.list

## Cell 8: 학습 Config 생성

Fine-tuning용 커스텀 YAML config를 생성합니다.
기본 `cosyvoice3.yaml`에서 LR, epoch 등을 조정합니다.

In [ ]:
import yaml
import os

TRAIN_BASE = '/content/CosyVoice/examples/libritts/cosyvoice3'
CONF_DIR = os.path.join(TRAIN_BASE, 'conf')
os.makedirs(CONF_DIR, exist_ok=True)

# 기존 config가 있으면 읽어서 수정, 없으면 직접 생성
base_config_path = os.path.join(CONF_DIR, 'cosyvoice3.yaml')
finetune_config_path = os.path.join(CONF_DIR, 'cosyvoice3_finetune.yaml')

if os.path.exists(base_config_path):
    print(f'Base config found: {base_config_path}')
    print('Using base config with fine-tune overrides')
    # 기존 config 읽기
    with open(base_config_path, 'r') as f:
        content = f.read()
    print(content[:2000])
    print('...')
    print('\nBase config will be used directly. Fine-tuning params are passed via CLI.')
else:
    print('Base config not found - check if repo clone was successful')
    print('The training script will use CLI args for fine-tuning params')

## Cell 9: DeepSpeed Config

Colab T4 (16GB VRAM)에서 0.5B 모델을 학습하려면 DeepSpeed Stage 2가 필요합니다.

In [ ]:
import json

TRAIN_BASE = '/content/CosyVoice/examples/libritts/cosyvoice3'
CONF_DIR = os.path.join(TRAIN_BASE, 'conf')

# DeepSpeed Stage 2 config (메모리 절약)
ds_config = {
    "train_micro_batch_size_per_gpu": 1,
    "gradient_accumulation_steps": 2,
    "optimizer": {
        "type": "Adam",
        "params": {
            "lr": 5e-5,
            "betas": [0.9, 0.95],
            "weight_decay": 0.01
        }
    },
    "scheduler": {
        "type": "WarmupCosineLR",
        "params": {
            "warmup_min_lr": 1e-6,
            "warmup_max_lr": 5e-5,
            "warmup_num_steps": 100,
            "total_num_steps": 10000
        }
    },
    "fp16": {
        "enabled": True
    },
    "zero_optimization": {
        "stage": 2,
        "offload_optimizer": {
            "device": "cpu",
            "pin_memory": True
        },
        "allgather_partitions": True,
        "allgather_bucket_size": 5e8,
        "overlap_comm": True,
        "reduce_scatter": True,
        "reduce_bucket_size": 5e8,
        "contiguous_gradients": True
    },
    "gradient_clipping": 5.0,
    "steps_per_print": 10,
    "wall_clock_breakdown": False
}

ds_path = os.path.join(CONF_DIR, 'ds_stage2.json')
with open(ds_path, 'w') as f:
    json.dump(ds_config, f, indent=2)

print(f'DeepSpeed config saved: {ds_path}')
print(json.dumps(ds_config, indent=2))

## Cell 10: Dry Run (1 Epoch 검증)

본격 학습 전에 1 epoch만 돌려서 확인:
- `<|endofprompt|>` 태그가 토크나이저에서 정상 인식되는지
- 스타일 태그 (`[excited]` 등)가 텍스트에 포함되어 학습되는지
- loss가 정상 출력되는지
- OOM 없이 GPU 메모리가 안정적인지

In [ ]:
%%bash
cd /content/CosyVoice

export CUDA_VISIBLE_DEVICES="0"
export PYTHONPATH="/content/CosyVoice:$PYTHONPATH"

PRETRAINED="pretrained_models/Fun-CosyVoice3-0.5B"
TRAIN_BASE="examples/libritts/cosyvoice3"
CONFIG="$TRAIN_BASE/conf/cosyvoice3.yaml"

# Config 파일 선택 (커스텀 있으면 사용)
if [ -f "$TRAIN_BASE/conf/cosyvoice3_finetune.yaml" ]; then
    CONFIG="$TRAIN_BASE/conf/cosyvoice3_finetune.yaml"
fi

echo "=== DRY RUN: 1 epoch ==="
echo "Config: $CONFIG"
echo ""

# 1 epoch만 돌려서 검증 (max_epoch=1 override)
torchrun --nnodes=1 --nproc_per_node=1 \
    --rdzv_id=1986 --rdzv_backend="c10d" --rdzv_endpoint="localhost:1234" \
  cosyvoice/bin/train.py \
  --train_engine torch_ddp \
  --config $CONFIG \
  --train_data $TRAIN_BASE/data/train.data.list \
  --cv_data $TRAIN_BASE/data/dev.data.list \
  --model llm \
  --checkpoint $PRETRAINED/llm.pt \
  --model_dir $TRAIN_BASE/exp/dryrun/llm \
  --tensorboard_dir $TRAIN_BASE/tensorboard/dryrun \
  --ddp.dist_backend nccl \
  --num_workers 2 \
  --prefetch 100 \
  --pin_memory \
  --use_amp \
  --override_config "train_conf.max_epoch=1,train_conf.log_interval=1" \
  2>&1 | tail -50

echo ""
echo "=== Dry Run 완료 ==="
echo "GPU 메모리 사용량:"
nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader

## Cell 11: LLM SFT 학습 (본 학습)

### 학습 설정
- LR: 5e-5 + Cosine Decay (warmup 100 steps)
- Max Epoch: 50 (실제 30-40에서 best checkpoint 예상)
- LLM 파트만 학습 (Flow/HiFiGAN freeze)
- DeepSpeed Stage 2 (T4 16GB 대응)
- 5분 주기 자동 백업

In [ ]:
import threading
import time
import shutil

# 5분 주기 자동 백업 (학습 중 Colab 끊김 대비)
stop_backup = threading.Event()

def auto_backup():
    EXP_DIR = '/content/CosyVoice/examples/libritts/cosyvoice3/exp/cosyvoice3_ft/llm'
    BACKUP_EXP = '/content/drive/MyDrive/cosyvoice3_backup/exp'
    
    while not stop_backup.is_set():
        time.sleep(300)  # 5분
        if os.path.exists(EXP_DIR):
            try:
                # 최신 checkpoint만 백업 (전체 복사는 느림)
                checkpoints = sorted(
                    [f for f in os.listdir(EXP_DIR) if f.endswith('.pt')],
                    key=lambda x: os.path.getmtime(os.path.join(EXP_DIR, x)),
                    reverse=True
                )
                if checkpoints:
                    os.makedirs(BACKUP_EXP, exist_ok=True)
                    latest = checkpoints[0]
                    src = os.path.join(EXP_DIR, latest)
                    dst = os.path.join(BACKUP_EXP, latest)
                    if not os.path.exists(dst):
                        shutil.copy2(src, dst)
                        print(f'[Backup] {latest} -> Drive')
            except Exception as e:
                print(f'[Backup Error] {e}')

backup_thread = threading.Thread(target=auto_backup, daemon=True)
backup_thread.start()
print('Auto backup started (5min interval)')

In [ ]:
%%bash
cd /content/CosyVoice

export CUDA_VISIBLE_DEVICES="0"
export PYTHONPATH="/content/CosyVoice:$PYTHONPATH"

PRETRAINED="pretrained_models/Fun-CosyVoice3-0.5B"
TRAIN_BASE="examples/libritts/cosyvoice3"
CONFIG="$TRAIN_BASE/conf/cosyvoice3.yaml"

if [ -f "$TRAIN_BASE/conf/cosyvoice3_finetune.yaml" ]; then
    CONFIG="$TRAIN_BASE/conf/cosyvoice3_finetune.yaml"
fi

echo "=== LLM SFT Training ==="
echo "Config: $CONFIG"
echo "Pretrained: $PRETRAINED"
echo ""

torchrun --nnodes=1 --nproc_per_node=1 \
    --rdzv_id=1986 --rdzv_backend="c10d" --rdzv_endpoint="localhost:1234" \
  cosyvoice/bin/train.py \
  --train_engine torch_ddp \
  --config $CONFIG \
  --train_data $TRAIN_BASE/data/train.data.list \
  --cv_data $TRAIN_BASE/data/dev.data.list \
  --model llm \
  --checkpoint $PRETRAINED/llm.pt \
  --model_dir $TRAIN_BASE/exp/cosyvoice3_ft/llm/torch_ddp \
  --tensorboard_dir $TRAIN_BASE/tensorboard/cosyvoice3_ft/llm/torch_ddp \
  --ddp.dist_backend nccl \
  --num_workers 2 \
  --prefetch 100 \
  --pin_memory \
  --use_amp \
  --deepspeed_config $TRAIN_BASE/conf/ds_stage2.json \
  --deepspeed.save_states model+optimizer \
  --override_config "train_conf.max_epoch=50,train_conf.log_interval=10,train_conf.save_per_step=-1" \
  2>&1 | tee /content/train_log.txt

In [ ]:
# 백업 스레드 중지
stop_backup.set()

# 학습 로그에서 loss 추출해서 시각화
import re
import matplotlib.pyplot as plt

train_losses = []
cv_losses = []

log_path = '/content/train_log.txt'
if os.path.exists(log_path):
    with open(log_path, 'r') as f:
        for line in f:
            # train loss
            m = re.search(r'train_loss\s*[=:]\s*([\d.]+)', line)
            if m:
                train_losses.append(float(m.group(1)))
            # cv loss
            m = re.search(r'cv_loss\s*[=:]\s*([\d.]+)', line)
            if m:
                cv_losses.append(float(m.group(1)))

if train_losses:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(train_losses)
    axes[0].set_title('Train Loss')
    axes[0].set_xlabel('Log Step')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True)
    
    if cv_losses:
        axes[1].plot(cv_losses, color='orange')
        axes[1].set_title('CV Loss')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Loss')
        axes[1].grid(True)
        
        # Early stopping 경고
        if len(cv_losses) >= 3 and cv_losses[-1] > cv_losses[-2] > cv_losses[-3]:
            print('*** WARNING: CV loss increasing for 3 consecutive epochs - consider stopping! ***')
    
    plt.tight_layout()
    plt.savefig('/content/training_curves.png', dpi=100)
    plt.show()
    
    print(f'Train loss: {train_losses[0]:.4f} -> {train_losses[-1]:.4f}')
    if cv_losses:
        print(f'CV loss: {cv_losses[0]:.4f} -> {cv_losses[-1]:.4f} (best: {min(cv_losses):.4f})')
else:
    print('No training loss found in log')

## Cell 12: Checkpoint 평균화 + 평가

In [ ]:
%%bash
cd /content/CosyVoice

export PYTHONPATH="/content/CosyVoice:$PYTHONPATH"

TRAIN_BASE="examples/libritts/cosyvoice3"
EXP_DIR="$TRAIN_BASE/exp/cosyvoice3_ft/llm/torch_ddp"

# Top-5 checkpoint 평균화 (val_best 기준)
echo "=== Checkpoint Averaging (Top 5) ==="
python cosyvoice/bin/average_model.py \
    --dst_model $EXP_DIR/llm_avg.pt \
    --src_path $EXP_DIR \
    --num 5 \
    --val_best

echo ""
ls -la $EXP_DIR/llm_avg.pt

In [ ]:
import sys
sys.path.insert(0, '/content/CosyVoice')

import torch
import soundfile as sf
from IPython.display import Audio, display

# CosyVoice3 모델 로드 (평균화된 checkpoint 사용)
from cosyvoice.cli.cosyvoice import CosyVoice3

PRETRAINED = '/content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B'
EXP_DIR = '/content/CosyVoice/examples/libritts/cosyvoice3/exp/cosyvoice3_ft/llm/torch_ddp'

# 파인튜닝된 LLM checkpoint를 pretrained 디렉토리에 복사
import shutil
shutil.copy2(f'{EXP_DIR}/llm_avg.pt', f'{PRETRAINED}/llm.pt.bak')  # 원본 백업
shutil.copy2(f'{EXP_DIR}/llm_avg.pt', f'{PRETRAINED}/llm.pt')      # 교체

# 모델 로드
cosyvoice = CosyVoice3(PRETRAINED, load_jit=False, load_trt=False)
print('Model loaded')

In [ ]:
# 테스트 문장으로 음성 생성
test_sentences = [
    ("나한텐 일상이었어", "[excited]"),
    ("미안해, 다음엔 더 잘할게", "[sad]"),
    ("오~ 꽤하잖아!", "[happy]"),
    ("한숨 돌렸다 가자고. 시간은 충분하니까", "[calm]"),
    ("안녕하세요, 데비입니다. 오늘도 좋은 하루 되세요.", "[calm]"),
]

# 레퍼런스 오디오 (학습 데이터에서 하나 선택)
# spk2embedding에서 speaker embedding을 사용하므로 레퍼런스가 필요 없을 수 있음
# CosyVoice3의 inference_instruct2 사용

output_dir = '/content/test_outputs'
os.makedirs(output_dir, exist_ok=True)

for i, (text, style) in enumerate(test_sentences):
    print(f'\n--- Test {i+1}: {style} "{text}" ---')
    
    instruction = f'You are a helpful assistant.{style}<|endofprompt|>'
    
    # Debi
    for speaker in ['debi', 'marlene']:
        try:
            # zero-shot with speaker embedding
            ref_wav = f'/content/cosyvoice3_data/train/wavs/{speaker}_good_1_01.wav'
            if not os.path.exists(ref_wav):
                # 첫 번째 wav 파일 사용
                wavs_dir = f'/content/cosyvoice3_data/train/wavs/'
                spk_wavs = [f for f in os.listdir(wavs_dir) if f.startswith(speaker)]
                if spk_wavs:
                    ref_wav = os.path.join(wavs_dir, spk_wavs[0])
            
            output_wav = None
            for result in cosyvoice.inference_instruct2(
                text,
                instruction,
                ref_wav,
                stream=False,
            ):
                output_wav = result['tts_speech'].numpy()
            
            if output_wav is not None:
                out_path = f'{output_dir}/test_{i+1}_{speaker}_{style.strip("[]")}.wav'
                sf.write(out_path, output_wav.flatten(), 24000)
                print(f'  {speaker}: {out_path}')
                display(Audio(output_wav.flatten(), rate=24000))
        except Exception as e:
            print(f'  {speaker}: ERROR - {e}')

## Cell 13: 체크포인트 저장 (Drive + HuggingFace Hub)

In [ ]:
import shutil

# Google Drive에 최종 모델 저장
PRETRAINED = '/content/CosyVoice/pretrained_models/Fun-CosyVoice3-0.5B'
SAVE_DIR = '/content/drive/MyDrive/cosyvoice3_backup/final_model'

os.makedirs(SAVE_DIR, exist_ok=True)

# 추론에 필요한 파일만 복사
files_to_save = [
    'llm.pt',                     # 파인튜닝된 LLM
    'flow.pt',                     # 프리트레인 Flow (그대로)
    'hift.pt',                     # 프리트레인 HiFiGAN (그대로)
    'campplus.onnx',               # Speaker embedding
    'speech_tokenizer_v3.onnx',    # Speech tokenizer
    'cosyvoice.yaml',              # Config (있으면)
]

for fname in files_to_save:
    src = os.path.join(PRETRAINED, fname)
    if os.path.exists(src):
        shutil.copy2(src, os.path.join(SAVE_DIR, fname))
        size_mb = os.path.getsize(src) / (1024*1024)
        print(f'  Saved: {fname} ({size_mb:.1f} MB)')

# Qwen tokenizer 디렉토리도 복사
qwen_dirs = [d for d in os.listdir(PRETRAINED) if 'qwen' in d.lower() or 'tokenizer' in d.lower()]
for d in qwen_dirs:
    src = os.path.join(PRETRAINED, d)
    if os.path.isdir(src):
        dst = os.path.join(SAVE_DIR, d)
        if os.path.exists(dst):
            shutil.rmtree(dst)
        shutil.copytree(src, dst)
        print(f'  Saved dir: {d}')

print(f'\nSaved to: {SAVE_DIR}')

In [ ]:
# HuggingFace Hub에 업로드
from huggingface_hub import HfApi, login

# HF 토큰 입력
# login()  # 또는 아래에 토큰 직접 입력
# login(token="hf_YOUR_TOKEN")

HF_REPO = "2R4mi/cosyvoice3-debi-marlene"  # 원하는 repo 이름

api = HfApi()

# repo 생성 (없으면)
try:
    api.create_repo(repo_id=HF_REPO, exist_ok=True, private=True)
    print(f'Repo ready: {HF_REPO}')
except Exception as e:
    print(f'Repo creation: {e}')

# 파일 업로드
SAVE_DIR = '/content/drive/MyDrive/cosyvoice3_backup/final_model'

api.upload_folder(
    folder_path=SAVE_DIR,
    repo_id=HF_REPO,
    commit_message="CosyVoice3 fine-tuned: debi + marlene (LLM SFT)",
)
print(f'Uploaded to: https://huggingface.co/{HF_REPO}')